# 🎓 ASAG Grading Model Training

Fine-tune DeBERTa-v3-base for Automatic Short Answer Grading.

**Input:** `[CLS] question [SEP] reference_answer [SEP] student_answer [SEP]`

**Output:** 3-way classification (correct / partially_correct / incorrect)

**Data:** data-generate.csv (10,000 samples, 7,000 train)

After training, the model is pushed to Hugging Face Hub for free inference.

In [ ]:
# Step 0: Install dependencies
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn pandas

In [ ]:
# Step 1: Login to Hugging Face (needed to push model)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Step 2: Upload your data-generate.csv to Colab
# Option A: Upload manually via the file browser (left panel)
# Option B: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set the path to your CSV file
# Change this to match where you put the file:
CSV_PATH = '/content/drive/MyDrive/data-generate.csv'

# Or if you uploaded directly to Colab:
# CSV_PATH = '/content/data-generate.csv'

In [ ]:
# Step 3: Load and prepare data
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution (label_3way):")
print(df['label_3way'].value_counts())
print(f"\nSplit distribution:")
print(df['split'].value_counts())

In [ ]:
# Step 4: Prepare train/val/test splits
# Use the pre-defined splits from the CSV

# Map label_3way to integers
LABEL_MAP = {'correct': 0, 'partially_correct': 1, 'incorrect': 2}
LABEL_NAMES = ['correct', 'partially_correct', 'incorrect']

# Filter rows with valid label_3way
df_valid = df[df['label_3way'].isin(LABEL_MAP.keys())].copy()
df_valid['label_id'] = df_valid['label_3way'].map(LABEL_MAP)

# Use pre-defined splits
train_df = df_valid[df_valid['split'] == 'train']
val_df = df_valid[df_valid['split'] == 'valid']
test_df = df_valid[df_valid['split'].str.startswith('test')]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nTrain label distribution:")
print(train_df['label_3way'].value_counts())

In [ ]:
# Step 5: Create HuggingFace Dataset
from datasets import Dataset, DatasetDict

def make_input_text(row):
    """Format: question [SEP] reference_answer [SEP] student_answer"""
    q = str(row['question']) if pd.notna(row['question']) else ''
    r = str(row['reference_answer']) if pd.notna(row['reference_answer']) else ''
    s = str(row['student_answer']) if pd.notna(row['student_answer']) else ''
    return f"{q} [SEP] {r} [SEP] {s}"

def df_to_dataset(dataframe):
    texts = [make_input_text(row) for _, row in dataframe.iterrows()]
    labels = dataframe['label_id'].tolist()
    return Dataset.from_dict({'text': texts, 'label': labels})

dataset = DatasetDict({
    'train': df_to_dataset(train_df),
    'validation': df_to_dataset(val_df),
    'test': df_to_dataset(test_df),
})

print(dataset)
print(f"\nExample input: {dataset['train'][0]['text'][:200]}...")

In [ ]:
# Step 6: Tokenize
from transformers import AutoTokenizer

MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256,
    )

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print(tokenized)

In [ ]:
# Step 7: Define model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'correct', 1: 'partially_correct', 2: 'incorrect'},
    label2id={'correct': 0, 'partially_correct': 1, 'incorrect': 2},
)

print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Step 8: Training configuration
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

# ─── CHANGE THIS to your HF username ───
HF_USERNAME = 'YOUR_HF_USERNAME'  # <-- CHANGE THIS
REPO_NAME = f'{HF_USERNAME}/asag-grading-deberta-v3'
# ────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,  # Use mixed precision on T4 GPU
    push_to_hub=True,
    hub_model_id=REPO_NAME,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Will push to: {REPO_NAME}")

In [ ]:
# Step 9: Train! (~15-20 min on T4 GPU)
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# Step 10: Evaluate on test set
print("📊 Evaluating on test set...")
results = trainer.evaluate(tokenized['test'])
print(f"\nTest Results:")
print(f"  Accuracy:    {results['eval_accuracy']:.4f}")
print(f"  Macro F1:    {results['eval_macro_f1']:.4f}")
print(f"  Weighted F1: {results['eval_weighted_f1']:.4f}")

In [ ]:
# Step 11: Push to Hugging Face Hub
print(f"📤 Pushing model to {REPO_NAME}...")
trainer.push_to_hub(
    commit_message="ASAG grading model: DeBERTa-v3-base fine-tuned on 7K samples"
)
tokenizer.push_to_hub(REPO_NAME)
print(f"✅ Model available at: https://huggingface.co/{REPO_NAME}")
print(f"\n🔗 Use in your demo app:")
print(f"   HF_MODEL_ID={REPO_NAME}")

In [ ]:
# Step 12: Quick test inference
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=REPO_NAME,
    tokenizer=REPO_NAME,
    device=0,
)

# Test examples
examples = [
    "What is photosynthesis? [SEP] Plants convert light energy into chemical energy stored in glucose. [SEP] Plants use sunlight to make food from CO2 and water.",
    "What is photosynthesis? [SEP] Plants convert light energy into chemical energy stored in glucose. [SEP] Plants get food from soil.",
    "What is photosynthesis? [SEP] Plants convert light energy into chemical energy stored in glucose. [SEP] I like biology class.",
]

print("\n🧪 Test predictions:")
for text in examples:
    result = classifier(text, truncation=True, max_length=256)
    student = text.split('[SEP]')[-1].strip()
    print(f"  '{student[:50]}...' → {result[0]['label']} ({result[0]['score']:.3f})")

## ✅ Done!

Your model is now on Hugging Face Hub. To use it in the demo app:

1. Copy your model ID (e.g., `your-username/asag-grading-deberta-v3`)
2. Set it as `HF_MODEL_ID` in your demo app's `.env.local`
3. The app will call HF Inference API with your custom model

**Free tier limits:** ~30K inference calls/month. For a thesis demo, this is more than enough.